In [ ]:
# ============================================================
#
# TRADING BOT V4.0
#
# KRAKEN + PAPER TRADING + RISIKOMANAGEMENT
# TRENDFOLGE: MA20/50/200 + ADX-Stärkefilter
# ATR-Positionsgröße + ATR-Trailing-Stop (kein festes Take-Profit)
#
# BTC / ETH / SOL
#
# Kontowährung: EUR
#
# ============================================================


# ============================================================
# 1. BIBLIOTHEKEN
# ============================================================

import requests
import pandas as pd

import json
import time
import math
import os
import re
import shutil
import traceback

from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

from google.colab import drive


# ------------------------------------------------------------
# AUSGABE-DESIGN (Farben, Kästen, Emojis für Google Colab)
# ------------------------------------------------------------

FARBE_GRUEN = "\033[92m"
FARBE_ROT = "\033[91m"
FARBE_GELB = "\033[93m"
FARBE_BLAU = "\033[94m"
FARBE_CYAN = "\033[96m"
FARBE_GRAU = "\033[90m"
FARBE_FETT = "\033[1m"
FARBE_ENDE = "\033[0m"

BREITE = 60


def faerben(text, farbe):

    return f"{farbe}{text}{FARBE_ENDE}"


def wert_faerben(zahl, text):

    # Grün bei >= 0, Rot bei < 0 - für Gewinn/Verlust-Angaben

    return faerben(
        text,
        FARBE_GRUEN if zahl >= 0 else FARBE_ROT
    )


def kasten(titel, emoji=""):

    linie = "═" * BREITE

    print("")

    print(faerben(linie, FARBE_BLAU))

    kopf = f"{emoji}  {titel}" if emoji else titel

    print(faerben(kopf.center(BREITE), FARBE_FETT))

    print(faerben(linie, FARBE_BLAU))


def unterkasten(titel, emoji=""):

    kopf = f"{emoji}  {titel}" if emoji else titel

    print("")

    print(faerben(kopf, FARBE_FETT))

    print(faerben("─" * BREITE, FARBE_GRAU))


def trenner():

    print(faerben("─" * BREITE, FARBE_GRAU))


def zeile(label, wert, emoji=None, breite_label=26):

    praefix = f"{emoji} " if emoji else ""

    print(
        f"{praefix}{label:<{breite_label}} {wert}"
    )


# --- Dashboard-Boxen (┌─┐ │ └─┘) für Portfolio/Positionen/Signale ---

INNEN_BREITE = 62

# Emojis/Sonderzeichen wirken in Monospace-Schriften (Colab)
# doppelt so breit wie ihre Länge in Python (len() zählt 1
# Zeichen). Für einigermaßen saubere Rahmen wird das hier grob
# nachgebildet - perfekte Pixel-Ausrichtung ist nicht das Ziel.
# ANSI-Farbcodes sind unsichtbar und zählen nicht zur Breite.

ANSI_MUSTER = re.compile(r"\033\[[0-9;]*m")


def sichtbare_breite(text):

    text_ohne_farbcodes = ANSI_MUSTER.sub("", text)

    breite = 0

    for zeichen in text_ohne_farbcodes:

        code = ord(zeichen)

        if code == 0xFE0F:

            continue

        if (
            0x1F300 <= code <= 0x1FAFF
            or 0x2600 <= code <= 0x27BF
            or code in (0x25B2, 0x25BC, 0x25CF)
        ):

            breite += 2

        else:

            breite += 1

    return breite


def box_start(titel, emoji=""):

    kopf_text = f"{emoji} {titel}" if emoji else titel

    kopf = f"┌─ {kopf_text} "

    # Gesamtbreite der oberen Rahmenlinie muss der Breite von
    # box_zeile()/box_ende() entsprechen: "│ " + INNEN_BREITE +
    # " │" bzw. "└" + "─"*(INNEN_BREITE+2) + "┘" = INNEN_BREITE+4

    rest = (
        INNEN_BREITE + 4
        - sichtbare_breite(kopf)
        - 1
    )

    if rest < 0:

        rest = 0

    print("")

    print(
        faerben(kopf, FARBE_BLAU)
        + faerben("─" * rest + "┐", FARBE_BLAU)
    )


def box_zeile(text=""):

    innen = sichtbare_breite(text)

    fuellung = max(
        0,
        INNEN_BREITE - innen
    )

    print(
        faerben("│ ", FARBE_BLAU)
        + text
        + " " * fuellung
        + faerben(" │", FARBE_BLAU)
    )


def spalte(text, breite):

    # ANSI-sicheres Auffüllen einer Tabellenspalte auf feste
    # sichtbare Breite - normales str.ljust() würde die
    # unsichtbaren Farbcodes mitzählen und die Spalte dadurch
    # zu schmal auffüllen.

    fehlend = max(
        0,
        breite - sichtbare_breite(text)
    )

    return text + " " * fehlend


def box_ende():

    print(
        faerben(
            "└" + "─" * (INNEN_BREITE + 2) + "┘",
            FARBE_BLAU
        )
    )


def balken(anteil, breite=20):

    anteil = max(0.0, min(1.0, anteil))

    gefuellt = round(anteil * breite)

    farbe = FARBE_GRUEN if anteil < 0.8 else FARBE_ROT

    fuellstrich = faerben("█" * gefuellt, farbe)

    leerstrich = faerben(
        "░" * (breite - gefuellt),
        FARBE_GRAU
    )

    return f"[{fuellstrich}{leerstrich}]"


def pfeil(wert):

    if wert > 0:

        return faerben("▲", FARBE_GRUEN)

    if wert < 0:

        return faerben("▼", FARBE_ROT)

    return faerben("─", FARBE_GRAU)


def signal_text(signal):

    if signal == "LONG":

        return faerben("🟢 LONG", FARBE_GRUEN)

    return faerben("⚪ NO TRADE", FARBE_GRAU)


kasten("TRADING BOT V4.0", "🤖")

print(
    faerben(
        "Kraken Paper Trading · Trendfolge-Strategie".center(BREITE),
        FARBE_GRAU
    )
)


# ============================================================
# 2. GOOGLE DRIVE VERBINDEN
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 3. GRUNDEINSTELLUNGEN
# ============================================================

STARTKAPITAL = 10000.00

# Ausschließlich Paper Trading
PAPER_TRADING = True

# Automatische virtuelle Käufe
AUTOMATISCHE_KAEUFE = True

# Kursüberwachung alle 5 Minuten
INTERVALL = 300

# Risikomanagement
MAX_RISIKO_PRO_TRADE = 0.01
MAX_GESAMTRISIKO = 0.03
MAX_POSITIONSANTEIL = 0.50

MAX_OFFENE_POSITIONEN = 3

MAX_TAGESVERLUST = 0.03

# Handelsstrategie: Trendfolge mit ATR-Positionsgröße
# und ATR-Trailing-Stop (kein fixes Take-Profit, siehe Analyse
# zur Kostenasymmetrie: fixe kleine Targets verlieren gegen
# Gebühren+Slippage, laufengelassene Trends tragen sie)

TREND_FILTER_PERIODE = 200      # langsamer Trendfilter (MA200)

ADX_PERIODE = 14
ADX_MINDESTWERT = 20            # Mindest-Trendstärke für ein Signal

ATR_PERIODE = 14
ATR_STOP_MULTIPLIKATOR = 2.0        # initialer Stop = Einstieg - 2x ATR
ATR_TRAILING_MULTIPLIKATOR = 3.0    # Trailing-Stop = höchster Schluss seit Einstieg - 3x ATR

# Mindestanzahl an Stundenkerzen, bevor gehandelt wird
# (Platz für MA200 + Warmlaufzeit von ATR/ADX)
MIN_ERFORDERLICHE_KERZEN = TREND_FILTER_PERIODE + 50

# Simulierte Handelskosten (Kraken: Taker ~0,26%, hier bewusst
# konservativer angesetzt inkl. Slippage-Puffer)
GEBUEHR = 0.001
SLIPPAGE = 0.002

# Handelspause nach Verkauf
HANDELSPAUSE = 86400

# Zeitintervall der historischen Kerzen
KERZEN_INTERVALL = 60

# Sekunden pro Stundenkerze
KERZEN_SEKUNDEN = 3600


# ============================================================
# 4. KRYPTOWÄHRUNGEN
# ============================================================

COINS = {

    "bitcoin": {
        "name": "Bitcoin",
        "paar": "XBT/EUR"
    },

    "ethereum": {
        "name": "Ethereum",
        "paar": "ETH/EUR"
    },

    "solana": {
        "name": "Solana",
        "paar": "SOL/EUR"
    }

}


# ============================================================
# 5. SPEICHERORTE
# ============================================================

ORDNER = Path(
    "/content/drive/MyDrive/Trading_Bot"
)

ORDNER.mkdir(
    parents=True,
    exist_ok=True
)

MARKTDATEN_ORDNER = ORDNER / "Marktdaten_Kraken"

MARKTDATEN_ORDNER.mkdir(
    parents=True,
    exist_ok=True
)

# Neues Kontoschema (Trendfolge/ATR/Trailing-Stop statt
# fixem Take-Profit) - eigene Datei, damit alte V3.1-Positionen
# (anderes Datenschema mit take_profit) nicht versehentlich
# geladen werden. Die bisherige konto_v2.json bleibt unangetastet.

KONTO_DATEI = ORDNER / "konto_v3_trendfolge.json"

# Sicherung der bisherigen Kontodatei

BACKUP_DATEI = ORDNER / "konto_v3_trendfolge_backup.json"


# ============================================================
# 6. HILFSFUNKTIONEN
# ============================================================

def zeitstempel():

    # Intern wird konsequent UTC gespeichert (Kontodatei,
    # Transaktionen, Kraken-Stundenkerzen) - das hält Tages-
    # limit-Reset, 24h-Handelspause und Kerzengrenzen robust
    # gegen die Zeitumstellung. Nur die Anzeige für den
    # Menschen läuft über lokale_zeit_anzeigen() unten.

    return datetime.now(
        timezone.utc
    ).isoformat()


BERLIN_ZEITZONE = ZoneInfo("Europe/Berlin")


def lokale_zeit_anzeigen(zeitpunkt_utc):

    # Wandelt einen UTC-Zeitpunkt für die Anzeige nach
    # Europe/Berlin um - mit dem korrekten deutschen Kürzel
    # (MESZ im Sommer, MEZ im Winter). datetime.strftime("%Z")
    # liefert dafür nur "CEST"/"CET" (englisch), daher hier
    # manuell über die Sommerzeit-Erkennung gesetzt.

    lokal = zeitpunkt_utc.astimezone(BERLIN_ZEITZONE)

    ist_sommerzeit = (
        lokal.dst() is not None
        and lokal.dst().total_seconds() > 0
    )

    kuerzel = "MESZ" if ist_sommerzeit else "MEZ"

    return lokal.strftime(f"%d.%m.%Y %H:%M:%S {kuerzel}")


def euro(wert):

    text = f"{wert:,.2f}"

    text = (
        text.replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    return text + " €"


def gueltige_zahl(wert):

    return (
        isinstance(wert, (int, float))
        and not isinstance(wert, bool)
        and math.isfinite(wert)
    )


# ============================================================
# 7. KRAKEN API
# ============================================================

KRAKEN_API = "https://api.kraken.com/0/public"

session = requests.Session()

# Kurze Pause zwischen API-Anfragen
LETZTE_API_ANFRAGE = 0


def kraken_anfrage(endpunkt, parameter=None):

    global LETZTE_API_ANFRAGE

    url = f"{KRAKEN_API}/{endpunkt}"

    for versuch in range(3):

        # Mindestens 1,5 Sekunden Abstand
        # zwischen zwei Kraken-Anfragen

        abstand = time.monotonic() - LETZTE_API_ANFRAGE

        if abstand < 1.5:

            time.sleep(1.5 - abstand)

        LETZTE_API_ANFRAGE = time.monotonic()

        try:

            response = session.get(
                url,
                params=parameter,
                timeout=20
            )

            # HTTP-Limit erreicht

            if response.status_code == 429:

                wartezeit = 30 * (versuch + 1)

                print(
                    f"Kraken API-Limit erreicht. "
                    f"Warte {wartezeit} Sekunden."
                )

                time.sleep(wartezeit)

                continue

            response.raise_for_status()

            daten = response.json()

            # Kraken meldet API-Fehler im JSON

            fehler = daten.get("error", [])

            if fehler:

                fehlermeldung = ", ".join(fehler)

                # Vorübergehende API-Probleme

                if any(
                    "Rate limit" in f
                    or "Service:Unavailable" in f
                    or "Temporary" in f
                    for f in fehler
                ):

                    time.sleep(30 * (versuch + 1))

                    continue

                raise RuntimeError(
                    f"Kraken API: {fehlermeldung}"
                )

            return daten["result"]

        except (
            requests.exceptions.Timeout,
            requests.exceptions.ConnectionError
        ):

            if versuch == 2:

                raise

            time.sleep(10 * (versuch + 1))

    raise RuntimeError(
        f"Kraken-Anfrage fehlgeschlagen: {endpunkt}"
    )


# ============================================================
# 8. KRAKEN-HANDELSPAARE ERMITTELN
# ============================================================

def handelspaare_ermitteln():

    print("")
    print("Prüfe verfügbare Kraken-Handelspaare ...")

    daten = kraken_anfrage("AssetPairs")

    ergebnis = {}

    for coin, info in COINS.items():

        gesucht = info["paar"]

        treffer = []

        for kennung, paar_info in daten.items():

            if paar_info.get("wsname") == gesucht:

                treffer.append(
                    (kennung, paar_info)
                )

        # Zusätzliche Märkte und Spezialpaare
        # dürfen nicht versehentlich ausgewählt werden.

        treffer = [
            (kennung, info)
            for kennung, info in treffer
            if ".d" not in kennung.lower()
        ]

        if len(treffer) != 1:

            raise RuntimeError(
                f"Handelspaar nicht eindeutig: {gesucht}"
            )

        kennung, paar_info = treffer[0]

        ergebnis[coin] = kennung

        print(
            f"{info['name']}: "
            f"{gesucht} ({kennung})"
        )

    return ergebnis


HANDELSPAARE = handelspaare_ermitteln()


# ============================================================
# 9. AKTUELLE KURSE ABRUFEN
# ============================================================

def aktuelle_kurse():

    parameter = {
        "pair": ",".join(HANDELSPAARE.values())
    }

    daten = kraken_anfrage(
        "Ticker",
        parameter
    )

    kurse = {}

    for coin, kennung in HANDELSPAARE.items():

        if kennung not in daten:

            raise RuntimeError(
                f"Ticker fehlt: {coin}"
            )

        # Kraken liefert:
        # a = aktueller Verkaufspreis / Ask
        # b = aktueller Kaufpreis / Bid
        # c = letzter gehandelter Preis

        kurs = float(
            daten[kennung]["c"][0]
        )

        if not gueltige_zahl(kurs) or kurs <= 0:

            raise ValueError(
                f"Ungültiger Kraken-Kurs: {coin}"
            )

        kurse[coin] = kurs

    return kurse


# ============================================================
# 10. HISTORISCHE KURSE SPEICHERN
# ============================================================

def kursdatei(coin):

    return (
        MARKTDATEN_ORDNER
        / f"{coin}_1h.csv"
    )


def kurse_laden(coin):

    datei = kursdatei(coin)

    if not datei.exists():

        return pd.DataFrame(
            columns=["open", "high", "low", "close", "volume"],
            index=pd.DatetimeIndex([], tz="UTC", name="zeit")
        )

    df = pd.read_csv(
        datei,
        parse_dates=["zeit"],
        index_col="zeit"
    )

    df.index = pd.to_datetime(
        df.index,
        utc=True
    )

    if df.index.has_duplicates:

        raise ValueError(
            f"Doppelte Zeitstempel in {datei}"
        )

    df = df.sort_index()

    return df


def kurse_speichern(coin, df):

    datei = kursdatei(coin)

    temporaer = Path(
        str(datei) + ".tmp"
    )

    df.to_csv(
        temporaer,
        index_label="zeit"
    )

    os.replace(
        temporaer,
        datei
    )


# ============================================================
# 11. HISTORISCHE KURSE AKTUALISIEREN
# ============================================================

def historische_kurse(coin):

    df_alt = kurse_laden(coin)

    parameter = {

        "pair": HANDELSPAARE[coin],

        "interval": KERZEN_INTERVALL

    }

    # Bereits gespeicherte Daten nicht erneut
    # vollständig herunterladen.

    if not df_alt.empty:

        letzter_zeitpunkt = int(
            df_alt.index[-1].timestamp()
        )

        # Letzte bekannte Kerze nochmals abrufen,
        # damit Übergänge überprüft werden können.

        parameter["since"] = (
            letzter_zeitpunkt - KERZEN_SEKUNDEN
        )

    daten = kraken_anfrage(
        "OHLC",
        parameter
    )

    kennung = HANDELSPAARE[coin]

    if kennung not in daten:

        raise RuntimeError(
            f"OHLC-Daten fehlen: {coin}"
        )

    kerzen = daten[kennung]

    zeilen = []

    aktuelle_stunde = (
        pd.Timestamp.now(tz="UTC").floor("h")
    )

    for kerze in kerzen:

        zeit = pd.to_datetime(
            int(kerze[0]),
            unit="s",
            utc=True
        )

        # Noch laufende Stundenkerze ausschließen

        if zeit >= aktuelle_stunde:

            continue

        zeilen.append({

            "zeit": zeit,

            "open": float(kerze[1]),

            "high": float(kerze[2]),

            "low": float(kerze[3]),

            "close": float(kerze[4]),

            "volume": float(kerze[6])

        })

    if not zeilen:

        raise RuntimeError(
            f"Keine abgeschlossenen Kerzen: {coin}"
        )

    df_neu = pd.DataFrame(
        zeilen
    ).set_index("zeit")

    # Bereits gespeicherte und neue Kurse
    # zusammenführen.

    df = pd.concat([
        df_alt,
        df_neu
    ])

    df = df[
        ~df.index.duplicated(keep="last")
    ]

    df = df.sort_index()

    # Nur vollständige Stunden speichern

    df = df[
        df.index < aktuelle_stunde
    ]

    # Daten plausibilisieren

    for spalte in [
        "open",
        "high",
        "low",
        "close"
    ]:

        if (
            df[spalte].isna().any()
            or (df[spalte] <= 0).any()
        ):

            raise ValueError(
                f"Ungültige historische Kurse: {coin}"
            )

    # Neue Daten dauerhaft speichern

    kurse_speichern(
        coin,
        df
    )

    print(
        f"{COINS[coin]['name']}: "
        f"{len(df)} Stundenkerzen gespeichert."
    )

    return df


# ============================================================
# 12. TECHNISCHE INDIKATOREN (ATR, ADX)
# ============================================================

def true_range_berechnen(df):

    vorheriger_schluss = df["close"].shift(1)

    bereich = pd.concat([

        df["high"] - df["low"],

        (df["high"] - vorheriger_schluss).abs(),

        (df["low"] - vorheriger_schluss).abs()

    ], axis=1).max(axis=1)

    return bereich


def atr_berechnen(df, periode=ATR_PERIODE):

    tr = true_range_berechnen(df)

    atr = tr.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    return atr


def adx_berechnen(df, periode=ADX_PERIODE):

    hoch_diff = df["high"].diff()

    tief_diff = -df["low"].diff()

    plus_dm = hoch_diff.where(
        (hoch_diff > tief_diff) & (hoch_diff > 0),
        0.0
    )

    minus_dm = tief_diff.where(
        (tief_diff > hoch_diff) & (tief_diff > 0),
        0.0
    )

    tr = true_range_berechnen(df)

    tr_geglaettet = tr.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    plus_dm_geglaettet = plus_dm.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    minus_dm_geglaettet = minus_dm.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    plus_di = 100 * (
        plus_dm_geglaettet / tr_geglaettet
    )

    minus_di = 100 * (
        minus_dm_geglaettet / tr_geglaettet
    )

    dx = 100 * (
        (plus_di - minus_di).abs()
        / (plus_di + minus_di)
    )

    adx = dx.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    return adx


# ============================================================
# 13. MARKTDATEN VALIDIEREN
# ============================================================

def marktdaten_pruefen(df, coin):

    # Genug historische Kerzen für MA200 + Indikator-Warmlauf

    if len(df) < MIN_ERFORDERLICHE_KERZEN:

        raise ValueError(
            f"Zu wenige historische Kurse: {coin}"
        )

    # Die letzten 50 Stunden müssen lückenlos sein.

    letzte_kerzen = df.tail(50)

    abstaende = (
        letzte_kerzen.index.to_series()
        .diff()
        .dropna()
    )

    if not (
        abstaende == pd.Timedelta(hours=1)
    ).all():

        raise ValueError(
            f"Lücke in den historischen Kursdaten: {coin}"
        )

    # Keine Handelssignale aus veralteten Daten

    aktuelle_stunde = (
        pd.Timestamp.now(tz="UTC").floor("h")
    )

    letzter_zeitpunkt = df.index[-1]

    if letzter_zeitpunkt < (
        aktuelle_stunde - pd.Timedelta(hours=1)
    ):

        raise ValueError(
            f"Historische Kurse veraltet: {coin}"
        )

    return True


# ============================================================
# 14. HANDELSKONTO
# ============================================================

def neues_konto():

    return {

        "startkapital": STARTKAPITAL,

        "guthaben": STARTKAPITAL,

        "positionen": {},

        "transaktionen": [],

        "erstellt_am": zeitstempel(),

        "tagesdatum": None,

        "tagesstartwert": STARTKAPITAL,

        "tageslimit_erreicht": False,

        "letzte_analyse": None,

        "letzter_ausstieg": {}

    }


def konto_pruefen(konto):

    if not isinstance(konto, dict):

        raise ValueError(
            "Ungültige Kontodatei."
        )

    for feld in [
        "startkapital",
        "guthaben",
        "positionen",
        "transaktionen"
    ]:

        if feld not in konto:

            raise ValueError(
                f"Kontofeld fehlt: {feld}"
            )

    if (
        not gueltige_zahl(konto["guthaben"])
        or konto["guthaben"] < -0.000001
    ):

        raise ValueError(
            "Ungültiges Kontoguthaben."
        )

    if not isinstance(
        konto["positionen"], dict
    ):

        raise ValueError(
            "Ungültige Positionsdaten."
        )

    if not isinstance(
        konto["transaktionen"], list
    ):

        raise ValueError(
            "Ungültige Transaktionshistorie."
        )

    for coin, position in konto["positionen"].items():

        if coin not in COINS:

            raise ValueError(
                f"Unbekannte Kryptowährung: {coin}"
            )

        for feld in [
            "menge",
            "einstieg",
            "stop_loss",
            "gesamtkosten",
            "hoechster_schlusskurs",
            "atr_bei_einstieg"
        ]:

            wert = position.get(feld)

            if (
                not gueltige_zahl(wert)
                or wert <= 0
            ):

                raise ValueError(
                    f"Ungültige Position: {coin}, {feld}"
                )

        if not (
            position["stop_loss"]
            < position["einstieg"]
        ):

            raise ValueError(
                f"Ungültige Kursmarken: {coin}"
            )


def konto_speichern(konto):

    konto_pruefen(konto)

    temporaer = Path(
        str(KONTO_DATEI) + ".tmp"
    )

    with open(
        temporaer,
        "w"
    ) as datei:

        json.dump(
            konto,
            datei,
            indent=4,
            ensure_ascii=False
        )

    os.replace(
        temporaer,
        KONTO_DATEI
    )


def konto_laden():

    if KONTO_DATEI.exists():

        with open(
            KONTO_DATEI,
            "r"
        ) as datei:

            konto = json.load(datei)

        konto_pruefen(konto)

        return konto

    konto = neues_konto()

    konto_speichern(konto)

    return konto


# ============================================================
# 15. BESTEHENDES KONTO ÜBERNEHMEN
# ============================================================

if (
    KONTO_DATEI.exists()
    and not BACKUP_DATEI.exists()
):

    shutil.copy2(
        KONTO_DATEI,
        BACKUP_DATEI
    )

    print(
        "Sicherung des bisherigen Kontos erstellt."
    )


konto = konto_laden()

konto.setdefault("tagesdatum", None)

konto.setdefault(
    "tagesstartwert",
    STARTKAPITAL
)

konto.setdefault(
    "tageslimit_erreicht",
    False
)

konto.setdefault(
    "letzte_analyse",
    None
)

konto.setdefault(
    "letzter_ausstieg",
    {}
)

# Verkaufszeitpunkte aus der bisherigen
# Transaktionshistorie übernehmen.

for transaktion in konto["transaktionen"]:

    if transaktion.get("aktion") == "VERKAUF":

        coin = transaktion.get("coin")

        verkaufszeit = transaktion.get("zeit")

        if coin in COINS and verkaufszeit:

            bisher = konto["letzter_ausstieg"].get(coin)

            if (
                bisher is None
                or verkaufszeit > bisher
            ):

                konto["letzter_ausstieg"][coin] = (
                    verkaufszeit
                )


konto_speichern(konto)

unterkasten("Handelskonto geladen", "💾")

zeile("Guthaben", euro(konto["guthaben"]))

zeile("Offene Positionen", str(len(konto["positionen"])))


# ============================================================
# 16. KONTOVERMÖGEN
# ============================================================

def kontowert_berechnen(konto, kurse):

    wert = konto["guthaben"]

    for coin, position in konto["positionen"].items():

        wert += (
            position["menge"]
            * kurse[coin]
        )

    return wert


# ============================================================
# 17. RISIKO OFFENER POSITIONEN
# ============================================================

def offenes_risiko_berechnen(konto):

    gesamtrisiko = 0.0

    for position in konto["positionen"].values():

        stop_erloes = (

            position["menge"]

            * position["stop_loss"]

            * (1 - SLIPPAGE)

            * (1 - GEBUEHR)

        )

        risiko = max(
            0,
            position["gesamtkosten"] - stop_erloes
        )

        gesamtrisiko += risiko

    return gesamtrisiko


# ============================================================
# 18. RISK GATE
# ============================================================

def risk_gate(konto, coin, kurse, atr_wert):

    # Keine echten Trades

    if not PAPER_TRADING:

        return None, "Nur Paper Trading erlaubt"

    if not AUTOMATISCHE_KAEUFE:

        return None, "Automatische Käufe deaktiviert"

    if konto.get(
        "tageslimit_erreicht",
        False
    ):

        return None, "Tagesverlustlimit erreicht"

    # Keine doppelte Position

    if coin in konto["positionen"]:

        return None, "Position bereits offen"

    if (
        len(konto["positionen"])
        >= MAX_OFFENE_POSITIONEN
    ):

        return None, "Maximale Positionszahl erreicht"

    # Handelspause nach Verkauf

    letzter_ausstieg = konto[
        "letzter_ausstieg"
    ].get(coin)

    if letzter_ausstieg:

        zeitpunkt = datetime.fromisoformat(
            letzter_ausstieg
        )

        if zeitpunkt.tzinfo is None:

            zeitpunkt = zeitpunkt.replace(
                tzinfo=timezone.utc
            )

        if (
            datetime.now(timezone.utc)
            - zeitpunkt
        ).total_seconds() < HANDELSPAUSE:

            return None, "24 Stunden Handelspause"

    # ATR muss gültig sein

    if (
        not gueltige_zahl(atr_wert)
        or atr_wert <= 0
    ):

        return None, "Ungültiger ATR-Wert"

    # Gesamtes Kontovermögen

    kapital = kontowert_berechnen(
        konto,
        kurse
    )

    # Bereits gebundenes Risiko

    offenes_risiko = offenes_risiko_berechnen(
        konto
    )

    # Verfügbares Gesamtrisiko

    verfuegbares_risiko = (

        kapital * MAX_GESAMTRISIKO

        - offenes_risiko

    )

    # Maximales Risiko je Trade

    risiko_budget = min(

        kapital * MAX_RISIKO_PRO_TRADE,

        verfuegbares_risiko

    )

    if risiko_budget <= 0:

        return None, "Kein Risikobudget"

    # Simulierter Einstiegskurs

    kurs = kurse[coin]

    einstieg = kurs * (1 + SLIPPAGE)

    # ATR-basierter initialer Stop

    stop_loss = einstieg - (
        ATR_STOP_MULTIPLIKATOR * atr_wert
    )

    if stop_loss <= 0:

        return None, "Ungültiger ATR-Stop"

    # Risiko je Einheit inklusive Kosten

    stop_erloes = (

        stop_loss

        * (1 - SLIPPAGE)

        * (1 - GEBUEHR)

    )

    risiko_je_einheit = (

        einstieg * (1 + GEBUEHR)

        - stop_erloes

    )

    if risiko_je_einheit <= 0:

        return None, "Ungültige Risikoberechnung"

    # Positionsgröße nach Risiko

    menge_risiko = (

        risiko_budget

        / risiko_je_einheit

    )

    # Positionsgröße nach maximalem Positionswert

    menge_positionslimit = (

        kapital * MAX_POSITIONSANTEIL

    ) / (

        einstieg * (1 + GEBUEHR)

    )

    # Verfügbares Guthaben berücksichtigen

    menge_guthaben = (

        konto["guthaben"]

    ) / (

        einstieg * (1 + GEBUEHR)

    )

    menge = min(

        menge_risiko,

        menge_positionslimit,

        menge_guthaben

    )

    if (
        not gueltige_zahl(menge)
        or menge <= 0
    ):

        return None, "Ungültige Positionsgröße"

    geplantes_risiko = (
        menge * risiko_je_einheit
    )

    # Abschließende Risikoprüfung

    if geplantes_risiko > (
        kapital * MAX_RISIKO_PRO_TRADE + 1e-8
    ):

        return None, "Einzelrisiko überschritten"

    if (
        offenes_risiko + geplantes_risiko
        > kapital * MAX_GESAMTRISIKO + 1e-8
    ):

        return None, "Gesamtrisiko überschritten"

    return {

        "menge": menge,

        "einstieg": einstieg,

        "stop_loss": stop_loss,

        "atr_bei_einstieg": atr_wert,

        "hoechster_schlusskurs": kurs,

        "geplantes_risiko": geplantes_risiko

    }, "Freigegeben"


# ============================================================
# 19. VIRTUELLER KAUF
# ============================================================

def virtueller_kauf(konto, coin, pruefung):

    menge = pruefung["menge"]

    einstieg = pruefung["einstieg"]

    positionswert = (
        menge * einstieg
    )

    gebuehr = (
        positionswert * GEBUEHR
    )

    gesamtkosten = (
        positionswert + gebuehr
    )

    if coin in konto["positionen"]:

        print(
            faerben(
                "⚠️  Kauf abgelehnt: Position bereits offen.",
                FARBE_GELB
            )
        )

        return False

    if gesamtkosten > konto["guthaben"] + 1e-8:

        print(
            faerben(
                "⚠️  Kauf abgelehnt: Guthaben zu gering.",
                FARBE_GELB
            )
        )

        return False

    # Guthaben reduzieren

    konto["guthaben"] -= gesamtkosten

    # Position anlegen

    konto["positionen"][coin] = {

        "menge": menge,

        "einstieg": einstieg,

        "stop_loss": pruefung["stop_loss"],

        "atr_bei_einstieg": pruefung["atr_bei_einstieg"],

        "hoechster_schlusskurs": pruefung["hoechster_schlusskurs"],

        "gesamtkosten": gesamtkosten,

        "kaufzeit": zeitstempel(),

        "datenquelle": "Kraken"

    }

    # Transaktion dokumentieren

    konto["transaktionen"].append({

        "zeit": zeitstempel(),

        "coin": coin,

        "aktion": "KAUF",

        "menge": menge,

        "kurs": einstieg,

        "gebuehr": gebuehr,

        "gesamtkosten": gesamtkosten,

        "atr_bei_einstieg": pruefung["atr_bei_einstieg"],

        "datenquelle": "Kraken"

    })

    konto_speichern(konto)

    abstand_stop = einstieg - pruefung["stop_loss"]

    abstand_stop_prozent = (
        abstand_stop / einstieg * 100
    )

    box_start(f"KAUF: {COINS[coin]['name']}", "✅")

    box_zeile(f"Kaufkurs        {euro(einstieg)}")

    box_zeile(f"Positionswert   {euro(positionswert)}")

    box_zeile(
        f"Risiko          "
        f"{euro(pruefung['geplantes_risiko'])}"
    )

    box_zeile(
        f"Stop-Loss       {euro(pruefung['stop_loss'])}  "
        f"(-{euro(abstand_stop)} / -{abstand_stop_prozent:.2f} %)"
    )

    box_zeile("Take-Profit     kein festes Ziel (Trailing-Stop)")

    box_zeile(
        f"ATR bei Einstieg {euro(pruefung['atr_bei_einstieg'])}"
    )

    box_ende()

    return True


# ============================================================
# 20. VIRTUELLER VERKAUF
# ============================================================

def virtueller_verkauf(
    konto,
    coin,
    referenzkurs,
    grund
):

    position = konto["positionen"][coin]

    menge = position["menge"]

    verkaufskurs = (
        referenzkurs * (1 - SLIPPAGE)
    )

    verkaufswert = (
        menge * verkaufskurs
    )

    gebuehr = (
        verkaufswert * GEBUEHR
    )

    netto_erloes = (
        verkaufswert - gebuehr
    )

    gewinn = (
        netto_erloes
        - position["gesamtkosten"]
    )

    # Guthaben aktualisieren

    konto["guthaben"] += netto_erloes

    # Position entfernen

    del konto["positionen"][coin]

    # Zeitpunkt des Verkaufs speichern

    konto["letzter_ausstieg"][coin] = (
        zeitstempel()
    )

    # Transaktion dokumentieren

    konto["transaktionen"].append({

        "zeit": zeitstempel(),

        "coin": coin,

        "aktion": "VERKAUF",

        "grund": grund,

        "menge": menge,

        "kurs": verkaufskurs,

        "gebuehr": gebuehr,

        "netto_erloes": netto_erloes,

        "gewinn": gewinn,

        "datenquelle": "Kraken"

    })

    konto_speichern(konto)

    grund_text = {
        "STOP": "Trailing-Stop ausgelöst",
        "TAGESLIMIT": "Tagesverlustlimit erreicht"
    }.get(grund, grund)

    box_start(f"VERKAUF: {COINS[coin]['name']}", "🛑")

    box_zeile(f"Grund           {grund_text}")

    box_zeile(f"Verkaufskurs    {euro(verkaufskurs)}")

    box_zeile(
        "Gewinn/Verlust  "
        + wert_faerben(gewinn, euro(gewinn))
        + f"  {pfeil(gewinn)}"
    )

    box_ende()


# ============================================================
# 21. TRAILING-STOP AKTUALISIEREN
# ============================================================

def trailing_stop_berechnen(
    position,
    letzter_schluss,
    aktueller_atr
):

    # Reine Berechnung ohne Seiteneffekt (kein Speichern) -
    # so lässt sich die Trailing-Stop-Logik im Funktionstest
    # prüfen, ohne die echte Kontodatei anzufassen.

    hoechster = position["hoechster_schlusskurs"]

    stop = position["stop_loss"]

    if (
        not gueltige_zahl(letzter_schluss)
        or not gueltige_zahl(aktueller_atr)
        or aktueller_atr <= 0
    ):

        return hoechster, stop, False

    # Trailing-Anker nur nach oben anpassen

    if letzter_schluss > hoechster:

        hoechster = letzter_schluss

    neuer_stop = (

        hoechster

        - (ATR_TRAILING_MULTIPLIKATOR * aktueller_atr)

    )

    # Stop darf nur steigen, nie sinken

    if neuer_stop > stop:

        return hoechster, neuer_stop, True

    return hoechster, stop, False


def trailing_stop_aktualisieren(
    konto,
    coin,
    letzter_schluss,
    aktueller_atr
):

    if coin not in konto["positionen"]:

        return

    position = konto["positionen"][coin]

    alter_stop = position["stop_loss"]

    hoechster, neuer_stop, geaendert = trailing_stop_berechnen(
        position,
        letzter_schluss,
        aktueller_atr
    )

    position["hoechster_schlusskurs"] = hoechster

    if geaendert:

        position["stop_loss"] = neuer_stop

        konto_speichern(konto)

        print(
            faerben(
                f"  ↳ 🔧 Trailing-Stop {COINS[coin]['name']} "
                f"nachgezogen: {euro(alter_stop)} → "
                f"{euro(neuer_stop)}",
                FARBE_GELB
            )
        )


# ============================================================
# 22. POSITIONEN ÜBERWACHEN
# ============================================================

def positionen_ueberwachen(konto, kurse):

    if not konto["positionen"]:

        return

    ausgestoppte_coins = []

    box_start("Positions-Check", "🔍")

    box_zeile(
        f"{'Coin':<10}{'Kurs':<12}{'Stop-Loss':<13}"
        f"{'Abstand':<10}Status"
    )

    for coin in list(
        konto["positionen"].keys()
    ):

        position = konto["positionen"][coin]

        kurs = kurse[coin]

        stop = position["stop_loss"]

        abstand_stop_prozent = (
            (kurs - stop) / kurs * 100
            if kurs else 0.0
        )

        ausgestoppt = kurs <= stop

        if ausgestoppt:

            ausgestoppte_coins.append(coin)

            status = faerben("🛑 STOP", FARBE_ROT)

        else:

            status = faerben("✅ offen", FARBE_GRUEN)

        box_zeile(
            f"{COINS[coin]['name']:<10}"
            f"{euro(kurs):<12}"
            f"{euro(stop):<13}"
            f"{abstand_stop_prozent:>5.2f} %   "
            f"{status}"
        )

    box_ende()

    # Verkäufe nach der Tabellenanzeige ausführen, damit jede
    # ausgelöste Verkaufsbox darunter erscheint statt die
    # Übersicht zu unterbrechen.

    for coin in ausgestoppte_coins:

        virtueller_verkauf(
            konto,
            coin,
            kurse[coin],
            "STOP"
        )


# ============================================================
# 23. TAGESVERLUSTLIMIT
# ============================================================

def tageslimit_pruefen(konto, kurse):

    heute = datetime.now(
        timezone.utc
    ).date().isoformat()

    kapital = kontowert_berechnen(
        konto,
        kurse
    )

    # Tagesstartwert speichern

    if konto.get("tagesdatum") != heute:

        konto["tagesdatum"] = heute

        konto["tagesstartwert"] = kapital

        konto["tageslimit_erreicht"] = False

        konto_speichern(konto)

    verlustgrenze = (

        konto["tagesstartwert"]

        * (1 - MAX_TAGESVERLUST)

    )

    if kapital <= verlustgrenze:

        if not konto["tageslimit_erreicht"]:

            konto["tageslimit_erreicht"] = True

            konto_speichern(konto)

            kasten("TAGESVERLUSTLIMIT ERREICHT", "⚠️")

            print(
                faerben(
                    "Alle offenen Positionen werden "
                    "geschlossen. Kein Handel bis morgen.",
                    FARBE_ROT
                )
            )

            # Alle Positionen virtuell schließen

            for coin in list(
                konto["positionen"].keys()
            ):

                virtueller_verkauf(
                    konto,
                    coin,
                    kurse[coin],
                    "TAGESLIMIT"
                )

        return False

    return not konto["tageslimit_erreicht"]


# ============================================================
# 24. MARKTANALYSE
# ============================================================

def marktanalyse(konto, kurse):

    kasten("AUTOMATISCHE MARKTANALYSE", "📊")

    for coin, info in COINS.items():

        # Historische Kursdaten aktualisieren

        try:

            df = historische_kurse(coin)

            marktdaten_pruefen(
                df,
                coin
            )

        except (
            requests.exceptions.RequestException,
            RuntimeError,
            ValueError
        ) as e:

            box_start(info["name"], "📉")

            box_zeile(
                faerben(
                    f"Keine neuen Trades: {e}",
                    FARBE_GELB
                )
            )

            box_ende()

            continue

        # Indikatoren berechnen

        df["MA20"] = (
            df["close"]
            .rolling(20)
            .mean()
        )

        df["MA50"] = (
            df["close"]
            .rolling(50)
            .mean()
        )

        df["MA200"] = (
            df["close"]
            .rolling(TREND_FILTER_PERIODE)
            .mean()
        )

        df["ATR"] = atr_berechnen(
            df,
            ATR_PERIODE
        )

        df["ADX"] = adx_berechnen(
            df,
            ADX_PERIODE
        )

        aktuell = df.iloc[-1]

        ma20 = float(aktuell["MA20"])
        ma50 = float(aktuell["MA50"])
        ma200 = float(aktuell["MA200"])
        atr = float(aktuell["ATR"])
        adx = float(aktuell["ADX"])
        letzter_schluss = float(aktuell["close"])

        box_start(info["name"], "📈")

        if not all(
            gueltige_zahl(wert)
            for wert in [ma20, ma50, ma200, atr, adx]
        ):

            box_zeile(
                faerben(
                    "Indikatoren noch nicht vollständig "
                    "berechnet (Warmlaufphase).",
                    FARBE_GELB
                )
            )

            box_ende()

            continue

        kurs = kurse[coin]

        box_zeile(f"Kurs            {euro(kurs)}")

        box_zeile(
            f"MA20/50/200     {euro(ma20)} / "
            f"{euro(ma50)} / {euro(ma200)}"
        )

        box_zeile(f"ATR14 / ADX14   {euro(atr)} / {adx:.1f}")

        # Trailing-Stop einer bestehenden Position
        # unabhängig vom Signal aktualisieren

        trailing_stop_aktualisieren(
            konto,
            coin,
            letzter_schluss,
            atr
        )

        # Signalprüfung

        trend_intakt = ma50 > ma200

        crossover_long = (
            ma20 > ma50
            and kurs > ma20
        )

        trend_stark = adx >= ADX_MINDESTWERT

        if (
            trend_intakt
            and crossover_long
            and trend_stark
        ):

            signal = "LONG"

        else:

            signal = "NO TRADE"

        box_zeile(f"Signal          {signal_text(signal)}")

        if signal != "LONG":

            gruende = []

            if not trend_intakt:

                gruende.append("Trendfilter (MA50<=MA200)")

            if not crossover_long:

                gruende.append("kein Crossover")

            if not trend_stark:

                gruende.append(f"ADX < {ADX_MINDESTWERT}")

            box_zeile(f"  ↳ {', '.join(gruende)}")

            box_ende()

            continue

        box_ende()

        # Risikoprüfung

        pruefung, grund = risk_gate(
            konto,
            coin,
            kurse,
            atr
        )

        if pruefung is None:

            print(
                faerben(
                    f"  ⚠️  Kein Kauf ({info['name']}): {grund}",
                    FARBE_GELB
                )
            )

            continue

        # Virtuellen Kauf ausführen

        virtueller_kauf(
            konto,
            coin,
            pruefung
        )


# ============================================================
# 25. STATUSANZEIGE
# ============================================================

def status_anzeigen(konto, kurse):

    kapital = kontowert_berechnen(
        konto,
        kurse
    )

    gewinn = (
        kapital - konto["startkapital"]
    )

    rendite = (
        gewinn
        / konto["startkapital"]
        * 100
    )

    offenes_risiko = offenes_risiko_berechnen(konto)

    risiko_anteil = (
        offenes_risiko / kapital
        if kapital else 0.0
    )

    # --- Portfolio-Übersicht ---

    box_start("Portfolio", "💰")

    box_zeile(f"Gesamtvermögen     {euro(kapital)}")

    box_zeile(
        "Gewinn/Verlust     "
        + wert_faerben(gewinn, f"{euro(gewinn)}")
        + f"  ({rendite:+.2f} %)  {pfeil(gewinn)}"
    )

    box_zeile(f"Guthaben (frei)    {euro(konto['guthaben'])}")

    box_zeile(
        f"Offene Positionen  "
        f"{len(konto['positionen'])} / {MAX_OFFENE_POSITIONEN}"
    )

    box_zeile(
        f"Risiko genutzt     "
        f"{balken(risiko_anteil / MAX_GESAMTRISIKO)} "
        f"{risiko_anteil * 100:.2f} % / "
        f"{MAX_GESAMTRISIKO * 100:.0f} %"
    )

    box_ende()

    # --- Positionstabelle ---

    if konto["positionen"]:

        box_start("Offene Positionen", "📋")

        box_zeile(
            f"{'Coin':<10}{'Kurs':<11}{'Einstieg':<11}"
            f"{'P&L':<9}{'Stop-Loss':<11}Abstand"
        )

        for coin, position in konto["positionen"].items():

            kurs = kurse[coin]

            stop = position["stop_loss"]

            abstand_stop_prozent = (
                (kurs - stop) / kurs * 100
                if kurs else 0.0
            )

            unrealisiert_prozent = (
                (kurs - position["einstieg"])
                / position["einstieg"] * 100
            )

            box_zeile(
                spalte(COINS[coin]["name"], 10)
                + spalte(euro(kurs), 11)
                + spalte(euro(position["einstieg"]), 11)
                + spalte(
                    wert_faerben(
                        unrealisiert_prozent,
                        f"{unrealisiert_prozent:+.2f} %"
                    ),
                    9
                )
                + spalte(euro(stop), 11)
                + f"{abstand_stop_prozent:.2f} %"
            )

        box_ende()

        print(
            faerben(
                "ℹ️  Take-Profit: kein festes Ziel — "
                "Exit ausschließlich über Trailing-Stop",
                FARBE_GRAU
            )
        )


# ============================================================
# 26. AUTOMATISCHER FUNKTIONSTEST
# ============================================================

def smoke_test():

    unterkasten("Funktionstest", "🧪")

    # --- Test 1: Risk Gate / Positionsberechnung ---

    testkonto = neues_konto()

    testkurse = {

        "bitcoin": 70000.0,

        "ethereum": 2500.0,

        "solana": 120.0

    }

    test_atr = 1200.0

    pruefung, grund = risk_gate(
        testkonto,
        "bitcoin",
        testkurse,
        test_atr
    )

    if pruefung is None:

        raise RuntimeError(
            f"Risk-Gate-Test fehlgeschlagen: {grund}"
        )

    assert pruefung["menge"] > 0

    assert pruefung["stop_loss"] < pruefung["einstieg"]

    assert pruefung["geplantes_risiko"] <= (

        STARTKAPITAL
        * MAX_RISIKO_PRO_TRADE
        + 1e-8

    )

    print(faerben("  ✅ Risk Gate", FARBE_GRUEN))

    print(faerben("  ✅ Positionsberechnung", FARBE_GRUEN))

    # --- Test 2: ATR / ADX auf synthetischen Daten ---

    anzahl = 300

    index = pd.date_range(
        "2025-01-01",
        periods=anzahl,
        freq="h",
        tz="UTC"
    )

    # Als NumPy-Array erzeugen, nicht als pd.Series mit eigenem
    # RangeIndex - sonst richtet Pandas beim DataFrame-Bau nach
    # Index aus (RangeIndex vs. DatetimeIndex) und alles wird NaN.

    basis = (
        pd.RangeIndex(anzahl).to_numpy(dtype=float)
        * 5
        + 50000
    )

    test_df = pd.DataFrame({

        "open": basis,

        "high": basis + 80,

        "low": basis - 80,

        "close": basis + 20,

        "volume": 1.0

    }, index=index)

    test_atr_serie = atr_berechnen(
        test_df,
        ATR_PERIODE
    )

    test_adx_serie = adx_berechnen(
        test_df,
        ADX_PERIODE
    )

    letzter_atr = float(
        test_atr_serie.iloc[-1]
    )

    letzter_adx = float(
        test_adx_serie.iloc[-1]
    )

    assert gueltige_zahl(letzter_atr) and letzter_atr > 0

    assert gueltige_zahl(letzter_adx) and 0 <= letzter_adx <= 100

    print(faerben("  ✅ ATR-Berechnung", FARBE_GRUEN))

    print(faerben("  ✅ ADX-Berechnung", FARBE_GRUEN))

    # --- Test 3: Trailing-Stop zieht nur nach oben nach ---
    # Reine Funktion ohne Speichern - die echte Kontodatei
    # bleibt beim Funktionstest unberührt.

    testposition = {

        "menge": 0.01,

        "einstieg": 70000.0,

        "stop_loss": 67600.0,

        "atr_bei_einstieg": 1200.0,

        "hoechster_schlusskurs": 70000.0,

        "gesamtkosten": 700.0

    }

    hoechster, neuer_stop, geaendert = trailing_stop_berechnen(
        testposition,
        letzter_schluss=72000.0,
        aktueller_atr=1000.0
    )

    assert geaendert is True

    assert neuer_stop > 67600.0

    testposition["hoechster_schlusskurs"] = hoechster

    testposition["stop_loss"] = neuer_stop

    # Fallender Kurs darf den Stop nicht zurückziehen

    hoechster2, neuer_stop2, geaendert2 = trailing_stop_berechnen(
        testposition,
        letzter_schluss=69000.0,
        aktueller_atr=1000.0
    )

    assert geaendert2 is False

    assert neuer_stop2 == neuer_stop

    print(faerben("  ✅ Trailing-Stop-Logik", FARBE_GRUEN))


# ============================================================
# 27. INITIALISIERUNG
# ============================================================

if not PAPER_TRADING:

    raise RuntimeError(
        "Dieses Programm darf nur virtuell handeln."
    )


# Zuerst alle Kernfunktionen überprüfen

smoke_test()


# ============================================================
# 28. AUTOMATISCHER BETRIEB
# ============================================================

kasten("TRADING BOT V4.0 GESTARTET", "🚀")

zeile("Handelsmodus", "PAPER TRADING", "📝")
zeile("Datenquelle", "Kraken", "🔗")
zeile("Kontowährung", "EUR", "💶")
zeile(
    "Strategie",
    "Trendfolge (MA20/50/200, ADX, ATR-Trailing-Stop)",
    "📈"
)
zeile("Marktanalyse", "stündlich", "🕐")
zeile("Positionsüberwachung", "alle 5 Minuten", "🔍")
zeile(
    "Automatische Käufe",
    "aktiv" if AUTOMATISCHE_KAEUFE else "deaktiviert",
    "🤖"
)

trenner()


# ============================================================
# 29. HAUPTSCHLEIFE
# ============================================================

try:

    while True:

        try:

            # Intern wird mit UTC gerechnet (Tageslimit-Reset,
            # 24h-Handelspause, Kerzengrenzen) - für die Anzeige
            # wird nur die Textausgabe nach Europe/Berlin
            # (MESZ/MEZ) umgerechnet.

            jetzt = datetime.now(
                timezone.utc
            )

            kasten(
                f"KURSPRÜFUNG · {lokale_zeit_anzeigen(jetzt)}",
                "🕒"
            )

            # ----------------------------------------
            # 1. Konto laden
            # ----------------------------------------

            konto = konto_laden()

            # ----------------------------------------
            # 2. Aktuelle Kraken-Kurse abrufen
            # ----------------------------------------

            kurse = aktuelle_kurse()

            # ----------------------------------------
            # 3. Tagesstartwert festhalten
            # ----------------------------------------

            heute = jetzt.date().isoformat()

            if konto.get("tagesdatum") != heute:

                konto["tagesdatum"] = heute

                konto["tagesstartwert"] = (
                    kontowert_berechnen(
                        konto,
                        kurse
                    )
                )

                konto["tageslimit_erreicht"] = False

                konto_speichern(konto)

            # ----------------------------------------
            # 4. Bestehende Positionen überwachen
            # ----------------------------------------

            positionen_ueberwachen(
                konto,
                kurse
            )

            # ----------------------------------------
            # 5. Tagesverlustlimit prüfen
            # ----------------------------------------

            handel_erlaubt = tageslimit_pruefen(
                konto,
                kurse
            )

            # ----------------------------------------
            # 6. Stündliche Marktanalyse
            # ----------------------------------------

            aktuelle_stunde = (
                jetzt.strftime("%Y-%m-%d-%H")
            )

            if (
                handel_erlaubt
                and konto.get("letzte_analyse")
                != aktuelle_stunde
            ):

                # Analysezeitpunkt speichern

                konto["letzte_analyse"] = (
                    aktuelle_stunde
                )

                konto_speichern(konto)

                # Marktanalyse durchführen

                marktanalyse(
                    konto,
                    kurse
                )

            # ----------------------------------------
            # 7. Kontostand anzeigen
            # ----------------------------------------

            status_anzeigen(
                konto,
                kurse
            )

            print(
                faerben(
                    "\nNächste Prüfung in ca. 5 Minuten.",
                    FARBE_GRAU
                )
            )

        except requests.exceptions.RequestException as e:

            print(
                faerben(f"\n⚠️  API-FEHLER: {e}", FARBE_GELB)
            )

            print(
                faerben(
                    "Keine Handelsausführung in diesem Durchlauf.",
                    FARBE_GELB
                )
            )

        except Exception as e:

            kasten("KRITISCHER PROGRAMMFEHLER", "🔥")

            traceback.print_exc()

            print(
                faerben(
                    "\nDer Bot wird aus Sicherheitsgründen "
                    "gestoppt.",
                    FARBE_ROT
                )
            )

            raise

        # Fünf Minuten warten

        time.sleep(INTERVALL)


except KeyboardInterrupt:

    kasten("TRADING BOT V4.0 GESTOPPT", "🛑")

    print(
        faerben(
            "Das virtuelle Handelskonto bleibt in "
            "Google Drive gespeichert.",
            FARBE_GRAU
        )
    )


════════════════════════════════════════════════════════════
                    🤖  TRADING BOT V4.0                     
════════════════════════════════════════════════════════════
        Kraken Paper Trading · Trendfolge-Strategie         
Mounted at /content/drive

Prüfe verfügbare Kraken-Handelspaare ...
Bitcoin: XBT/EUR (XXBTZEUR)
Ethereum: ETH/EUR (XETHZEUR)
Solana: SOL/EUR (SOLEUR)

💾  Handelskonto geladen
────────────────────────────────────────────────────────────
Guthaben                   9.906,98 €
Offene Positionen          0

🧪  Funktionstest
────────────────────────────────────────────────────────────
  ✅ Risk Gate
  ✅ Positionsberechnung
  ✅ ATR-Berechnung
  ✅ ADX-Berechnung
  ✅ Trailing-Stop-Logik

════════════════════════════════════════════════════════════
               🚀  TRADING BOT V4.0 GESTARTET                
════════════════════════════════════════════════════════════
📝 Handelsmodus               PAPER TRADING
🔗 Datenquelle                Kraken
💶 Kontowäh